# Sentinel AI: YOLO11 five-class retraining

This notebook trains the five classes that have labels in the current merged dataset: fire, smoke, accident, fallen_person, and crowd_anomaly. Road blockage remains a manual incident type; this detector does not claim to recognize it.

Keep the test split out of training and model selection. The held-out test evaluation runs only after training.

In [ ]:
!pip install -q ultralytics==8.4.140 pyyaml

In [ ]:
from pathlib import Path
from itertools import islice
import yaml
import torch

CLASS_NAMES = ['fire', 'smoke', 'accident', 'fallen_person', 'crowd_anomaly']
SOURCE_NAMES = CLASS_NAMES + ['road_blockage']

yaml_candidates = list(Path('/kaggle/input').rglob('data.yaml'))
matches = [path for path in yaml_candidates if path.parent.name == 'sentinel_dataset']
if len(matches) != 1:
    raise RuntimeError(f'Expected one sentinel_dataset/data.yaml, found: {matches}')
dataset_root = matches[0].parent
source_data = yaml.safe_load(matches[0].read_text(encoding='utf-8'))
if source_data.get('names') != SOURCE_NAMES:
    raise RuntimeError(f'Unexpected class order: {source_data.get("names")}')

split_dirs = {'train': 'train', 'val': 'valid', 'test': 'test'}
for split_name, split_dir in split_dirs.items():
    image_dir = dataset_root / split_dir / 'images'
    labels_dir = dataset_root / split_dir / 'labels'
    if not image_dir.is_dir() or not labels_dir.is_dir():
        raise FileNotFoundError(f'Missing {split_name} image/label folders')
    samples = list(islice(labels_dir.glob('*.txt'), 8))
    if not samples:
        raise RuntimeError(f'No labels found in {labels_dir}')
    for label in samples:
        for line_no, line in enumerate(label.open(errors='replace'), 1):
            fields = line.split()
            if not fields:
                continue
            if len(fields) != 5:
                raise RuntimeError(f'Malformed YOLO row: {label}:{line_no}')
            class_id = int(fields[0])
            if class_id not in range(5):
                raise RuntimeError(f'Unexpected class {class_id} at {label}:{line_no}; review labels before training')
            box = [float(value) for value in fields[1:]]
            if any(value < 0 or value > 1 for value in box):
                raise RuntimeError(f'Box value outside 0-1 at {label}:{line_no}')
    print(f'{split_name}: folders present; checked {len(samples)} sample labels')

kaggle_yaml = Path('/kaggle/working/sentinel_data_5class.yaml')
kaggle_yaml.write_text(yaml.safe_dump({
    'path': str(dataset_root),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 5,
    'names': CLASS_NAMES,
}, sort_keys=False), encoding='utf-8')
print(kaggle_yaml.read_text())
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before training.')

In [ ]:
import ultralytics
from ultralytics import YOLO

print('Ultralytics:', ultralytics.__version__)
model = YOLO('yolo11n.pt')
train_results = model.train(
    data=str(kaggle_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    seed=42,
    device=0,
    project='/kaggle/working/runs/detect',
    name='sentinel_yolo11_5class_v1',
    save_period=5,
)
best_path = Path(model.trainer.best)
print('Best checkpoint:', best_path)
if not best_path.is_file():
    raise FileNotFoundError(best_path)

## Held-out test evaluation

This evaluates the saved best checkpoint on the test split once. Do not use these test results to tune the model; tune against validation results and use the test split for final reporting.

In [ ]:
import shutil
from IPython.display import FileLink, display

best = YOLO(str(best_path))
assert list(best.names.values()) == CLASS_NAMES, best.names
test_metrics = best.val(data=str(kaggle_yaml), split='test', imgsz=640, batch=16, device=0)
print('Precision:', test_metrics.box.mp)
print('Recall:', test_metrics.box.mr)
print('mAP50:', test_metrics.box.map50)
print('mAP50-95:', test_metrics.box.map)

download_path = Path('/kaggle/working/sentinel_yolo11_5class.pt')
shutil.copy2(best_path, download_path)
display(FileLink(str(download_path)))

## Deploy the new weights locally

Download `sentinel_yolo11_5class.pt` and place it in `ai-service/models/`. The FastAPI service automatically prefers this five-class checkpoint when it exists. Until then, it loads the previous checkpoint and filters out its unsupported road-blockage output.

This five-class route does not guarantee stronger fallen-person or crowd-anomaly accuracy. Those classes remain data-limited and need more correctly labelled examples if the final test metrics remain weak.